In [1]:
!python -m tools.etl_review.view_parquet data/lakehouse/derived/derived.etf_aw_target_weight --schema --limit 20

path=data/lakehouse/derived/derived.etf_aw_target_weight
rows=85 columns=23 displayed=20

Schema:
                            column          dtype
                    schema_version         object
                  contract_version         object
                     calendar_name         object
                    rebalance_date         object
                    effective_date         object
                     strategy_name         object
                  strategy_version         object
                       sleeve_code         object
                       sleeve_role         object
                       risk_budget        float64
               volatility_estimate        float64
                  volatility_floor        float64
                 raw_target_weight        float64
         constrained_target_weight        float64
                     target_weight        float64
              target_weight_status         object
                    optimizer_name         object
  

In [2]:
!python -m tools.etl_review.view_parquet data/lakehouse/derived/derived.etf_aw_target_weight/2026/05/part-00000.parquet --limit 20

path=data/lakehouse/derived/derived.etf_aw_target_weight/2026/05/part-00000.parquet
rows=5 columns=23 displayed=5

Data:
         schema_version                 contract_version             calendar_name rebalance_date effective_date strategy_name             strategy_version sleeve_code  sleeve_role  risk_budget  volatility_estimate  volatility_floor  raw_target_weight  constrained_target_weight  target_weight target_weight_status       optimizer_name                                                                              optimizer_basis  turnover_estimate                                                                                                                                                                                                        quality_notes_json source_risk_budget_rebalance_date source_sleeve_daily_max_trade_date                ingested_at
etf_aw_target_weight_v1 etf_aw_target_weight_contract_v1 etf_aw_v1_monthly_post_20     2026-05-20     2026-05-20     

In [3]:
  import pandas as pd
  from pathlib import Path

  root = Path("data/lakehouse/derived/derived.etf_aw_target_weight")
  df = pd.concat([pd.read_parquet(p) for p in sorted(root.glob("*/*/part-00000.parquet"))])

  print("rows:", len(df))
  print("date range:", df["rebalance_date"].min(), df["rebalance_date"].max())
  print("\nstatus counts:")
  print(df["target_weight_status"].value_counts())

  latest_date = df["rebalance_date"].max()
  latest = df[df["rebalance_date"] == latest_date].sort_values("sleeve_role")

  display(latest[[
      "rebalance_date", "sleeve_role", "sleeve_code",
      "risk_budget", "volatility_estimate",
      "raw_target_weight", "constrained_target_weight", "target_weight",
      "target_weight_status", "turnover_estimate",
  ]])

  print("\nlatest sums:")
  print(latest[[
      "risk_budget", "raw_target_weight",
      "constrained_target_weight", "target_weight",
  ]].sum())

rows: 85
date range: 2025-01-20 2026-05-20

status counts:
target_weight_status
unavailable    75
complete       10
Name: count, dtype: int64


,rebalance_date,sleeve_role,sleeve_code,risk_budget,volatility_estimate,raw_target_weight,constrained_target_weight,target_weight,target_weight_status,turnover_estimate
0,2026-05-20,bond,511010.SH,0.191630,0.005000,0.322609,0.322609,0.322609,complete,0.051429
1,2026-05-20,cash,159001.SZ,0.179075,0.005000,0.301472,0.301472,0.301472,complete,0.051429
2,2026-05-20,equity_large,510300.SH,0.220925,0.010376,0.179220,0.179220,0.179220,complete,0.051429
3,2026-05-20,equity_small,159845.SZ,0.220925,0.015425,0.120562,0.120562,0.120562,complete,0.051429
4,2026-05-20,gold,518850.SH,0.187445,0.020723,0.076137,0.076137,0.076137,complete,0.051429



latest sums:
risk_budget                  1.0
raw_target_weight            1.0
constrained_target_weight    1.0
target_weight                1.0
dtype: float64
